
# Dublin Rental Market Lakehouse

## 1. Project Overview

This project aims to build an end-to-end data engineering pipeline using Databricks, PySpark, SQL, and Delta Lake.

The pipeline processes public rental market data from Dublin, Ireland, transforming raw datasets into structured and analytics-ready tables.

### Business Objectives

- Ingest public rental market datasets.
- Build a Lakehouse architecture using the Medallion approach.
- Apply data cleaning and standardization.
- Implement data quality checks.
- Generate analytical datasets for rental market analysis.

### Technology Stack

- Databricks Free Edition
- Apache Spark
- PySpark
- SQL
- Delta Lake
- Git and GitHub

---

## 2. Environment Validation

Before developing the data pipeline, we validate the Databricks environment and confirm that Spark can execute basic data processing operations.



### 2.1 Spark Session Validation

The first step is to verify the Spark version and test basic DataFrame creation and execution.


In [0]:

# Dublin Rental Market Lakehouse
# Step 1: Validate the Databricks environment

print("Dublin Rental Market Lakehouse")
print(f"Spark version: {spark.version}")

# Verify that Spark can process data
test_df = spark.range(1, 6)

display(test_df)


Dublin Rental Market Lakehouse
Spark version: 4.2.0


id
1
2
3
4
5



## 4. Source Data Acquisition

### 4.1 Dataset Description

The project uses the RTB Average Monthly Rent Report published by Ireland's Central Statistics Office (CSO).

The dataset provides statistical information about residential rental prices in Ireland.

### 4.2 Data Format

The original dataset is provided in JSON-stat format.

JSON-stat is a structured statistical data format containing dimensions, categories, metadata, and observations.

### 4.3 Data Ingestion Strategy

The original source file will be preserved without modifications.

The ingestion process will prioritize:

- Source data traceability.
- Schema and metadata inspection.
- Data integrity validation.
- Reproducible ingestion.


In [0]:
%sql
SELECT
    current_catalog() AS catalog_name,
    current_schema() AS schema_name;

catalog_name,schema_name
workspace,default



## 5. Unity Catalog Configuration

### 5.1 Database Organization

This section creates a dedicated schema for the Dublin Rental Market Lakehouse project.

Unity Catalog will be used to organize and manage the project's data assets.

The schema will contain the project's managed tables and storage volumes.


In [0]:
%sql

-- Create a dedicated schema for the project

CREATE SCHEMA IF NOT EXISTS workspace.dublin_rental_market;


In [0]:
%sql

-- Verify the project schema

SHOW SCHEMAS IN workspace;


databaseName
default
dublin_rental_market
information_schema



## 5. Storage Layer Configuration

### 5.1 Raw Data Storage

This section creates a dedicated storage location for the original source datasets.

A Unity Catalog Volume will be used to store raw files before they are processed by the data pipeline.

The original data will be preserved to ensure traceability, reproducibility, and data integrity.


In [0]:
%sql

-- Create a volume for raw source files

CREATE VOLUME IF NOT EXISTS
workspace.dublin_rental_market.raw_files;


In [0]:
%sql

-- Verify volume creation

SHOW VOLUMES IN workspace.dublin_rental_market;


database,volume_name
dublin_rental_market,raw_files


In [0]:

# Define the source file path
source_path = "/Volumes/workspace/dublin_rental_market/raw_files/rtb_average_monthly_rent.json"

# List files in the raw volume
files = dbutils.fs.ls(
    "/Volumes/workspace/dublin_rental_market/raw_files/"
)

# Display file information
for file in files:
    print(f"File name: {file.name}")
    print(f"File size: {file.size:,} bytes")


File name: rtb_average_monthly_rent.json
File size: 7,741,915 bytes


Before processing the source dataset, we inspect its internal structure using Python.

The objective is to identify:

- Dataset metadata and identifiers.
- Available dimensions and categories.
- Dimension sizes and labels.
- Observation values and data types.

This exploratory step helps define the appropriate data transformation strategy for the Bronze layer.

In [0]:

import json

# Load the original JSON-stat dataset
with open(source_path, "r", encoding="utf-8") as file:
    dataset = json.load(file)

# Display general dataset metadata
print("DATASET METADATA")
print("-" * 50)

print(f"Dataset class: {dataset.get('class')}")
print(f"Dataset ID: {dataset.get('id')}")
print(f"Dataset dimensions: {dataset.get('size')}")

# Inspect available dimensions
print("\nDATASET DIMENSIONS")
print("-" * 50)

dimensions = dataset.get("dimension", {})

for dimension_id, dimension_info in dimensions.items():

    category_index = dimension_info.get(
        "category", {}
    ).get("index", {})

    print(f"\nDimension: {dimension_id}")
    print(f"Label: {dimension_info.get('label')}")

    if isinstance(category_index, dict):
        print(f"Number of categories: {len(category_index)}")
    elif isinstance(category_index, list):
        print(f"Number of categories: {len(category_index)}")

# Inspect observations
print("\nOBSERVATION INFORMATION")
print("-" * 50)

observations = dataset.get("value", [])

print(f"Observation type: {type(observations).__name__}")
print(f"Number of observations: {len(observations)}")

# Display a small sample
if isinstance(observations, dict):
    sample = list(observations.items())[:5]
else:
    sample = observations[:5]

print(f"Sample observations: {sample}")


DATASET METADATA
--------------------------------------------------
Dataset class: dataset
Dataset ID: ['STATISTIC', 'TLIST(Q1)', 'C02970V03592', 'C02969V03591', 'C03004V03625']
Dataset dimensions: [1, 73, 7, 6, 446]

DATASET DIMENSIONS
--------------------------------------------------

Dimension: STATISTIC
Label: STATISTIC
Number of categories: 1

Dimension: TLIST(Q1)
Label: Quarter
Number of categories: 73

Dimension: C02970V03592
Label: Number of Bedrooms
Number of categories: 7

Dimension: C02969V03591
Label: Property Type
Number of categories: 6

Dimension: C03004V03625
Label: Location
Number of categories: 446

OBSERVATION INFORMATION
--------------------------------------------------
Observation type: list
Number of observations: 1367436
Sample observations: [805.3, None, None, 724.37, 588.23]


This step explores the category codes and labels associated with each dataset dimension.

The objective is to identify the available statistical indicators, reporting periods, bedroom categories, property types, and geographical locations.

Understanding these categories is essential to correctly transform the multidimensional JSON-stat structure into a tabular dataset.

In [0]:

# Inspect category codes and labels for each dimension

def inspect_dimension(dimension_id, dimension_info):

    category = dimension_info.get("category", {})
    labels = category.get("label", {})
    indexes = category.get("index", {})
    units = category.get("unit", {})

    print(f"\nDIMENSION: {dimension_id}")
    print(f"Description: {dimension_info.get('label')}")
    print("-" * 60)

    if isinstance(labels, dict):
        items = list(labels.items())
    elif isinstance(labels, list):
        items = list(enumerate(labels))
    else:
        items = []

    print(f"Total categories: {len(items)}")

    # Display all categories for smaller dimensions
    # Limit location output to the first 15 categories
    limit = 15 if dimension_id == "C03004V03625" else None

    for code, label in items[:limit]:
        print(f"Code: {code} | Label: {label}")

    if limit and len(items) > limit:
        print(f"... and {len(items) - limit} additional categories")

    if units:
        print(f"Unit metadata: {str(units)[:500]}")


# Inspect all dataset dimensions
for dimension_id, dimension_info in dimensions.items():
    inspect_dimension(dimension_id, dimension_info)



DIMENSION: STATISTIC
Description: STATISTIC
------------------------------------------------------------
Total categories: 1
Code: RIQ02 | Label: RTB Average Monthly Rent Report
Unit metadata: {'RIQ02': {'decimals': 2, 'label': 'Euro', 'position': 'end'}}

DIMENSION: TLIST(Q1)
Description: Quarter
------------------------------------------------------------
Total categories: 73
Code: 20074 | Label: 2007Q4
Code: 20081 | Label: 2008Q1
Code: 20082 | Label: 2008Q2
Code: 20083 | Label: 2008Q3
Code: 20084 | Label: 2008Q4
Code: 20091 | Label: 2009Q1
Code: 20092 | Label: 2009Q2
Code: 20093 | Label: 2009Q3
Code: 20094 | Label: 2009Q4
Code: 20101 | Label: 2010Q1
Code: 20102 | Label: 2010Q2
Code: 20103 | Label: 2010Q3
Code: 20104 | Label: 2010Q4
Code: 20111 | Label: 2011Q1
Code: 20112 | Label: 2011Q2
Code: 20113 | Label: 2011Q3
Code: 20114 | Label: 2011Q4
Code: 20121 | Label: 2012Q1
Code: 20122 | Label: 2012Q2
Code: 20123 | Label: 2012Q3
Code: 20124 | Label: 2012Q4
Code: 20131 | Label: 2013Q1
Co

JSON-stat stores observations in a flattened array. Each observation corresponds to a combination of dimension categories.

This step validates the category ordering and calculates the position of selected observations using the dataset dimension sizes.

The validation ensures that each rental value is associated with the correct quarter, bedroom category, property type, and location before the Bronze ingestion process.

In [0]:

from itertools import product
from math import prod

# Build ordered category metadata for each dimension
dimension_metadata = {}

for dimension_id, dimension_info in dimensions.items():

    category = dimension_info.get("category", {})
    labels = category.get("label", {})
    indexes = category.get("index", {})

    if isinstance(indexes, dict):
        ordered_codes = sorted(indexes, key=lambda code: indexes[code])
    elif isinstance(indexes, list):
        ordered_codes = indexes
    else:
        ordered_codes = list(labels.keys())

    dimension_metadata[dimension_id] = {
        "codes": ordered_codes,
        "labels": labels
    }

# Validate dimension sizes
dimension_sizes = dataset["size"]
dimension_ids = dataset["id"]

print("DIMENSION SIZE VALIDATION")
print("-" * 60)

for dimension_id, expected_size in zip(dimension_ids, dimension_sizes):

    actual_size = len(dimension_metadata[dimension_id]["codes"])

    print(
        f"{dimension_id}: "
        f"Expected = {expected_size}, "
        f"Actual = {actual_size}"
    )

    assert actual_size == expected_size, (
        f"Category count mismatch in {dimension_id}"
    )

# Calculate the flattened position of an observation
def calculate_flat_index(coordinates, sizes):

    position = 0

    for i, coordinate in enumerate(coordinates):
        stride = prod(sizes[i + 1:])
        position += coordinate * stride

    return position


# Validate the first five observation combinations
print("\nOBSERVATION MAPPING VALIDATION")
print("-" * 60)

sample_coordinates = list(
    product(*[range(size) for size in dimension_sizes])
)[:5]

for coordinates in sample_coordinates:

    flat_index = calculate_flat_index(
        coordinates,
        dimension_sizes
    )

    observation = observations[flat_index]

    print(f"\nFlattened position: {flat_index}")
    print(f"Observation value: {observation}")

    for dimension_id, coordinate in zip(
        dimension_ids, coordinates
    ):

        metadata = dimension_metadata[dimension_id]
        code = metadata["codes"][coordinate]
        label = metadata["labels"].get(code, code)

        print(f"{dimension_id}: {code} | {label}")


# Inspect missing observations
missing_count = sum(value is None for value in observations)

print("\nMISSING VALUE ANALYSIS")
print("-" * 60)
print(f"Total observations: {len(observations):,}")
print(f"Missing observations: {missing_count:,}")
print(
    f"Missing percentage: "
    f"{missing_count / len(observations) * 100:.2f}%"
)


DIMENSION SIZE VALIDATION
------------------------------------------------------------
STATISTIC: Expected = 1, Actual = 1
TLIST(Q1): Expected = 73, Actual = 73
C02970V03592: Expected = 7, Actual = 7
C02969V03591: Expected = 6, Actual = 6
C03004V03625: Expected = 446, Actual = 446

OBSERVATION MAPPING VALIDATION
------------------------------------------------------------

Flattened position: 0
Observation value: 805.3
STATISTIC: RIQ02 | RTB Average Monthly Rent Report
TLIST(Q1): 20074 | 2007Q4
C02970V03592: - | All bedrooms
C02969V03591: - | All property types
C03004V03625: 110000 | Carlow

Flattened position: 1
Observation value: None
STATISTIC: RIQ02 | RTB Average Monthly Rent Report
TLIST(Q1): 20074 | 2007Q4
C02970V03592: - | All bedrooms
C02969V03591: - | All property types
C03004V03625: 110200 | Carlow Town

Flattened position: 2
Observation value: None
STATISTIC: RIQ02 | RTB Average Monthly Rent Report
TLIST(Q1): 20074 | 2007Q4
C02970V03592: - | All bedrooms
C02969V03591: - | Al

## Bronze Layer - Raw Data Ingestion

### JSON-stat to PySpark DataFrame

The Bronze layer stores the source observations in a structured Delta table.

The transformation converts the flattened JSON-stat array into individual records, reconstructing the corresponding dimension categories using their original positions.

All observations are preserved, including missing values, to maintain source data integrity and support future data quality analysis.

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    ArrayType,
    DoubleType,
    IntegerType,
    StringType
)

# Define the source dataset dimensions
dimension_sizes = dataset["size"]
dimension_ids = dataset["id"]

# Create a Spark DataFrame containing the original observations
observations_schema = StructType([
    StructField(
        "observations",
        ArrayType(DoubleType()),
        nullable=False
    )
])

observations_df = spark.createDataFrame(
    [(observations,)],
    schema=observations_schema
)

# Expand the flattened observation array
bronze_df = observations_df.select(
    F.posexplode("observations").alias(
        "observation_index",
        "average_monthly_rent_eur"
    )
)

# Calculate the position of each category in every dimension
for i, dimension_id in enumerate(dimension_ids):

    stride = prod(dimension_sizes[i + 1:])
    dimension_size = dimension_sizes[i]

    bronze_df = bronze_df.withColumn(
        f"{dimension_id}_position",
        F.pmod(
            F.floor(
                F.col("observation_index") / F.lit(stride)
            ).cast("long"),
            F.lit(dimension_size)
        ).cast("int")
    )

# Create dimension lookup DataFrames and join them
for dimension_id in dimension_ids:

    metadata = dimension_metadata[dimension_id]

    category_rows = [
        (
            position,
            str(code),
            str(metadata["labels"].get(code, code))
        )
        for position, code in enumerate(metadata["codes"])
    ]

    lookup_df = spark.createDataFrame(
        category_rows,
        schema=StructType([
            StructField("category_position", IntegerType(), False),
            StructField("category_code", StringType(), False),
            StructField("category_label", StringType(), False)
        ])
    )

    bronze_df = (
        bronze_df
        .join(
            F.broadcast(lookup_df),
            bronze_df[f"{dimension_id}_position"]
            == lookup_df["category_position"],
            "left"
        )
        .drop("category_position")
        .withColumnRenamed("category_code", f"{dimension_id}_code")
        .withColumnRenamed("category_label", f"{dimension_id}_label")
    )

# Select the final Bronze columns
bronze_df = bronze_df.select(
    F.col("STATISTIC_code").alias("statistic_code"),
    F.col("STATISTIC_label").alias("statistic_name"),
    F.col("TLIST(Q1)_code").alias("quarter_code"),
    F.col("TLIST(Q1)_label").alias("quarter"),
    F.col("C02970V03592_code").alias("bedroom_code"),
    F.col("C02970V03592_label").alias("bedroom_category"),
    F.col("C02969V03591_code").alias("property_type_code"),
    F.col("C02969V03591_label").alias("property_type"),
    F.col("C03004V03625_code").alias("location_code"),
    F.col("C03004V03625_label").alias("location"),
    F.col("average_monthly_rent_eur"),
    F.current_timestamp().alias("ingested_at")
)

# Persist the Bronze dataset as a Delta table
bronze_table = (
    "workspace.dublin_rental_market.bronze_rental_observations"
)

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_table)
)

print(f"Bronze table created: {bronze_table}")


Bronze table created: workspace.dublin_rental_market.bronze_rental_observations


In [0]:

# Read the Bronze Delta table
bronze_table_df = spark.table(bronze_table)

# Display total records and missing values
bronze_table_df.select(
    F.count("*").alias("total_records"),
    F.count("average_monthly_rent_eur").alias("valid_observations"),
    F.sum(
        F.when(
            F.col("average_monthly_rent_eur").isNull(),
            1
        ).otherwise(0)
    ).alias("missing_observations")
).show()

# Display sample records
display(bronze_table_df.limit(10))

# Check the table schema
bronze_table_df.printSchema()


+-------------+------------------+--------------------+
|total_records|valid_observations|missing_observations|
+-------------+------------------+--------------------+
|      1367436|            376988|              990448|
+-------------+------------------+--------------------+



statistic_code,statistic_name,quarter_code,quarter,bedroom_code,bedroom_category,property_type_code,property_type,location_code,location,average_monthly_rent_eur,ingested_at
RIQ02,RTB Average Monthly Rent Report,20074,2007Q4,-,All bedrooms,-,All property types,110000,Carlow,805.3,2026-10-30T18:22:31.884Z
RIQ02,RTB Average Monthly Rent Report,20074,2007Q4,-,All bedrooms,-,All property types,110200,Carlow Town,null,2026-10-30T18:22:31.884Z
RIQ02,RTB Average Monthly Rent Report,20074,2007Q4,-,All bedrooms,-,All property types,110300,"Graiguecullen, Carlow",null,2026-10-30T18:22:31.884Z
RIQ02,RTB Average Monthly Rent Report,20074,2007Q4,-,All bedrooms,-,All property types,110500,"Tullow, Carlow",724.37,2026-10-30T18:22:31.884Z
RIQ02,RTB Average Monthly Rent Report,20074,2007Q4,-,All bedrooms,-,All property types,110550,Cavan,588.23,2026-10-30T18:22:31.884Z
RIQ02,RTB Average Monthly Rent Report,20074,2007Q4,-,All bedrooms,-,All property types,111100,Cavan Town,null,2026-10-30T18:22:31.884Z
RIQ02,RTB Average Monthly Rent Report,20074,2007Q4,-,All bedrooms,-,All property types,110600,"Bailieborough, Cavan",null,2026-10-30T18:22:31.884Z
RIQ02,RTB Average Monthly Rent Report,20074,2007Q4,-,All bedrooms,-,All property types,110800,"Ballyconnell, Cavan",null,2026-10-30T18:22:31.884Z
RIQ02,RTB Average Monthly Rent Report,20074,2007Q4,-,All bedrooms,-,All property types,110900,"Ballyjamesduff, Cavan",null,2026-10-30T18:22:31.884Z
RIQ02,RTB Average Monthly Rent Report,20074,2007Q4,-,All bedrooms,-,All property types,111000,"Belturbet, Cavan",null,2026-10-30T18:22:31.884Z


root
 |-- statistic_code: string (nullable = true)
 |-- statistic_name: string (nullable = true)
 |-- quarter_code: string (nullable = true)
 |-- quarter: string (nullable = true)
 |-- bedroom_code: string (nullable = true)
 |-- bedroom_category: string (nullable = true)
 |-- property_type_code: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- location_code: string (nullable = true)
 |-- location: string (nullable = true)
 |-- average_monthly_rent_eur: double (nullable = true)
 |-- ingested_at: timestamp (nullable = true)




## Silver Layer - Data Transformation

### Data Standardization and Quality

The Silver layer transforms Bronze records into a standardized dataset.

The transformation includes:

- Standardizing text fields and category labels.
- Converting rental values into decimal format.
- Extracting year and quarter from the reporting period.
- Creating a quarter reference date.
- Identifying missing and invalid rental values.
- Preserving all source records for traceability.

Missing rental values are retained and explicitly identified instead of being replaced with zero.


In [0]:

from pyspark.sql import functions as F

# Read the Bronze Delta table
silver_df = spark.table(bronze_table)

# Standardize text fields and transform data types
silver_df = (
    silver_df

    # Standardize category labels
    .withColumn("statistic_name", F.trim("statistic_name"))
    .withColumn("bedroom_category", F.trim("bedroom_category"))
    .withColumn("property_type", F.trim("property_type"))
    .withColumn("location", F.trim("location"))

    # Convert rental values to decimal
    .withColumn(
        "average_monthly_rent_eur",
        F.col("average_monthly_rent_eur").cast("decimal(18,2)")
    )

    # Extract year and quarter
    .withColumn(
        "year",
        F.substring("quarter", 1, 4).cast("integer")
    )
    .withColumn(
        "quarter_number",
        F.regexp_extract("quarter", r"Q([1-4])", 1).cast("integer")
    )

    # Create the quarter reference date
    .withColumn(
        "quarter_start_date",
        F.make_date(
            F.col("year"),
            ((F.col("quarter_number") - 1) * 3 + 1).cast("integer"),
            F.lit(1)
        )
    )

    # Create data quality indicators
    .withColumn(
        "is_missing_rent",
        F.col("average_monthly_rent_eur").isNull()
    )
    .withColumn(
        "is_valid_rent",
        F.col("average_monthly_rent_eur").isNotNull()
        & (F.col("average_monthly_rent_eur") > 0)
    )
)

# Define Silver table
silver_table = (
    "workspace.dublin_rental_market.silver_rental_observations"
)

# Save as Delta
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

print(f"Silver table created: {silver_table}")


Silver table created: workspace.dublin_rental_market.silver_rental_observations



### Silver Data Quality Validation

This validation evaluates record completeness, rental value validity, reporting period consistency, and duplicate dimensional combinations.

The objective is to verify that the transformation preserved the source dataset while producing standardized records suitable for analytical processing.


In [0]:

# Read the Silver Delta table
silver_table_df = spark.table(silver_table)

# Calculate data quality metrics
silver_table_df.select(
    F.count("*").alias("total_records"),

    F.sum(
        F.when(F.col("is_missing_rent"), 1).otherwise(0)
    ).alias("missing_rent"),

    F.sum(
        F.when(F.col("is_valid_rent"), 1).otherwise(0)
    ).alias("valid_rent"),

    F.sum(
        F.when(
            F.col("average_monthly_rent_eur") <= 0,
            1
        ).otherwise(0)
    ).alias("non_positive_rent"),

    F.sum(
        F.when(
            F.col("quarter_start_date").isNull(),
            1
        ).otherwise(0)
    ).alias("invalid_reporting_period")
).show()

# Check duplicate dimensional combinations
duplicate_count = (
    silver_table_df
    .groupBy(
        "statistic_code",
        "quarter_code",
        "bedroom_code",
        "property_type_code",
        "location_code"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate dimensional combinations: {duplicate_count}")

# Display sample standardized records
display(
    silver_table_df.select(
        "quarter",
        "quarter_start_date",
        "bedroom_category",
        "property_type",
        "location",
        "average_monthly_rent_eur",
        "is_missing_rent",
        "is_valid_rent"
    ).limit(10)
)


+-------------+------------+----------+-----------------+------------------------+
|total_records|missing_rent|valid_rent|non_positive_rent|invalid_reporting_period|
+-------------+------------+----------+-----------------+------------------------+
|      1367436|      990448|    376988|                0|                       0|
+-------------+------------+----------+-----------------+------------------------+

Duplicate dimensional combinations: 0


quarter,quarter_start_date,bedroom_category,property_type,location,average_monthly_rent_eur,is_missing_rent,is_valid_rent
2007Q4,2007-10-01,All bedrooms,All property types,Carlow,805.30,false,true
2007Q4,2007-10-01,All bedrooms,All property types,Carlow Town,null,true,false
2007Q4,2007-10-01,All bedrooms,All property types,"Graiguecullen, Carlow",null,true,false
2007Q4,2007-10-01,All bedrooms,All property types,"Tullow, Carlow",724.37,false,true
2007Q4,2007-10-01,All bedrooms,All property types,Cavan,588.23,false,true
2007Q4,2007-10-01,All bedrooms,All property types,Cavan Town,null,true,false
2007Q4,2007-10-01,All bedrooms,All property types,"Bailieborough, Cavan",null,true,false
2007Q4,2007-10-01,All bedrooms,All property types,"Ballyconnell, Cavan",null,true,false
2007Q4,2007-10-01,All bedrooms,All property types,"Ballyjamesduff, Cavan",null,true,false
2007Q4,2007-10-01,All bedrooms,All property types,"Belturbet, Cavan",null,true,false


##  Gold Layer - Analytical Data Products

### Geographical Scope Analysis

The source dataset contains 446 geographical categories across Ireland.

Before creating analytical tables, this step identifies Dublin-related geographical categories and examines the available geographical scope.

This prevents mixing different geographical levels when preparing rental market indicators.



In [0]:

# Identify Dublin and national geographical categories

location_metadata = dimension_metadata["C03004V03625"]

location_categories = [
    {
        "location_code": str(code),
        "location": str(location_metadata["labels"].get(code, code))
    }
    for code in location_metadata["codes"]
]

dublin_locations = [
    item for item in location_categories
    if "dublin" in item["location"].lower()
]

national_locations = [
    item for item in location_categories
    if any(
        term in item["location"].lower()
        for term in ["ireland", "national", "state"]
    )
]

print("DUBLIN LOCATIONS")
print("-" * 60)

for item in dublin_locations:
    print(f'{item["location_code"]} | {item["location"]}')

print("\nPOTENTIAL NATIONAL CATEGORIES")
print("-" * 60)

for item in national_locations:
    print(f'{item["location_code"]} | {item["location"]}')


DUBLIN LOCATIONS
------------------------------------------------------------
120500 | Dublin
120600 | Balbriggan, Dublin
120700 | Blackrock, Dublin
120800 | Booterstown, Dublin
120900 | Cabinteely, Dublin
121000 | Citywest, Dublin
121200 | Dalkey, Dublin
121300 | Donabate, Dublin
121400 | Dun Laoghaire, Dublin
121500 | Glenageary, Dublin
121600 | Howth, Dublin
121700 | Killiney, Dublin
121800 | Kinsealy, Dublin
121900 | Lucan, Dublin
122000 | Lusk, Dublin
122100 | Malahide, Dublin
122200 | Monkstown, Dublin
122300 | Mount Merrion, Dublin
122400 | Portmarnock, Dublin
122500 | Rathcoole, Dublin
122600 | Rush, Dublin
122700 | Saggart, Dublin
122800 | Sandycove, Dublin
122900 | Shankill, Dublin
123000 | Skerries, Dublin
123100 | Stepaside, Dublin
123200 | Stillorgan, Dublin
123300 | Swords, Dublin
123400 | Dublin 1
123500 | I.F.S.C., Dublin 1
123600 | Parnell Street, Dublin 1
123700 | Spencer Dock, Dublin 1
123800 | Summerhill, Dublin 1
123900 | Dublin 2
124000 | Aungier Street, Dublin 2



##  Gold Layer - Analytical Data Products

### Dublin Rental Market Analytics

The Gold layer contains business-oriented datasets derived from the validated Silver layer.

Four analytical products are created:

- Quarterly rental trends for Dublin.
- Rental values by property type.
- Rental values by bedroom category.
- Rental values across Dublin geographical categories.

The analytical datasets use valid rental observations and preserve the original CSO geographical identifiers.

Aggregated geographical categories are kept separate from more granular locations to prevent misleading comparisons and double counting.


In [0]:

from pyspark.sql import functions as F

# Read the validated Silver table
silver = spark.table(silver_table)

# Keep only valid rental observations
valid_silver = silver.filter(
    F.col("is_valid_rent") == True
)

# --------------------------------------------------
# 1. Dublin quarterly rental trends
# --------------------------------------------------

dublin_trends = (
    valid_silver
    .filter(
        (F.col("location_code") == "120500") &
        (F.col("bedroom_code") == "-") &
        (F.col("property_type_code") == "-")
    )
    .select(
        "location_code",
        "location",
        "quarter_code",
        "quarter",
        "quarter_start_date",
        "average_monthly_rent_eur"
    )
)

# --------------------------------------------------
# 2. Rental values by property type
# --------------------------------------------------

dublin_property_type = (
    valid_silver
    .filter(
        (F.col("location_code") == "120500") &
        (F.col("bedroom_code") == "-") &
        (F.col("property_type_code") != "-")
    )
    .select(
        "location_code",
        "location",
        "quarter",
        "quarter_start_date",
        "property_type_code",
        "property_type",
        "average_monthly_rent_eur"
    )
)

# --------------------------------------------------
# 3. Rental values by bedroom category
# --------------------------------------------------

dublin_bedrooms = (
    valid_silver
    .filter(
        (F.col("location_code") == "120500") &
        (F.col("bedroom_code") != "-") &
        (F.col("property_type_code") == "-")
    )
    .select(
        "location_code",
        "location",
        "quarter",
        "quarter_start_date",
        "bedroom_code",
        "bedroom_category",
        "average_monthly_rent_eur"
    )
)

# --------------------------------------------------
# 4. Rental values by Dublin geographical category
# --------------------------------------------------

# Identify Dublin-related categories from source metadata
dublin_location_codes = [
    str(code)
    for code in location_metadata["codes"]
    if (
        "dublin" in str(
            location_metadata["labels"].get(code, code)
        ).lower()
        and str(code) != "120500"
    )
]

dublin_location_detail = (
    valid_silver
    .filter(
        (F.col("location_code").isin(dublin_location_codes)) &
        (F.col("bedroom_code") == "-") &
        (F.col("property_type_code") == "-")
    )
    .select(
        "location_code",
        "location",
        "quarter",
        "quarter_start_date",
        "average_monthly_rent_eur"
    )
)

# --------------------------------------------------
# Save Gold Delta tables
# --------------------------------------------------

gold_tables = {
    "gold_dublin_rent_trends": dublin_trends,
    "gold_dublin_rent_by_property_type": dublin_property_type,
    "gold_dublin_rent_by_bedrooms": dublin_bedrooms,
    "gold_dublin_rent_by_location": dublin_location_detail
}

for table_name, dataframe in gold_tables.items():

    full_table_name = (
        f"workspace.dublin_rental_market.{table_name}"
    )

    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(full_table_name)
    )

    print(
        f"Created: {full_table_name} "
        f"| Records: {dataframe.count():,}"
    )


Created: workspace.dublin_rental_market.gold_dublin_rent_trends | Records: 73
Created: workspace.dublin_rental_market.gold_dublin_rent_by_property_type | Records: 365
Created: workspace.dublin_rental_market.gold_dublin_rent_by_bedrooms | Records: 438
Created: workspace.dublin_rental_market.gold_dublin_rent_by_location | Records: 10,628


In [0]:

# Validate all Gold analytical products

for table_name in gold_tables.keys():

    full_table_name = (
        f"workspace.dublin_rental_market.{table_name}"
    )

    gold_df = spark.table(full_table_name)

    print(f"\nTABLE: {table_name}")
    print("-" * 60)

    gold_df.select(
        F.count("*").alias("total_records"),
        F.countDistinct("quarter").alias("distinct_quarters"),
        F.min("average_monthly_rent_eur").alias("minimum_rent"),
        F.max("average_monthly_rent_eur").alias("maximum_rent")
    ).show()

# Inspect quarterly rental trends
print("\nDUBLIN QUARTERLY RENTAL TRENDS")
print("-" * 60)

display(
    spark.table(
        "workspace.dublin_rental_market.gold_dublin_rent_trends"
    )
    .orderBy("quarter_start_date")
)

# Inspect property type comparisons
print("\nRENTAL VALUES BY PROPERTY TYPE")
print("-" * 60)

display(
    spark.table(
        "workspace.dublin_rental_market.gold_dublin_rent_by_property_type"
    )
    .filter(F.col("quarter") == "2025Q4")
    .orderBy("average_monthly_rent_eur", ascending=False)
)

# Inspect bedroom category comparisons
print("\nRENTAL VALUES BY BEDROOM CATEGORY")
print("-" * 60)

display(
    spark.table(
        "workspace.dublin_rental_market.gold_dublin_rent_by_bedrooms"
    )
    .filter(F.col("quarter") == "2025Q4")
    .orderBy("average_monthly_rent_eur", ascending=False)
)



TABLE: gold_dublin_rent_trends
------------------------------------------------------------
+-------------+-----------------+------------+------------+
|total_records|distinct_quarters|minimum_rent|maximum_rent|
+-------------+-----------------+------------+------------+
|           73|               73|     1001.78|     2165.02|
+-------------+-----------------+------------+------------+


TABLE: gold_dublin_rent_by_property_type
------------------------------------------------------------
+-------------+-----------------+------------+------------+
|total_records|distinct_quarters|minimum_rent|maximum_rent|
+-------------+-----------------+------------+------------+
|          365|               73|      624.63|     3087.83|
+-------------+-----------------+------------+------------+


TABLE: gold_dublin_rent_by_bedrooms
------------------------------------------------------------
+-------------+-----------------+------------+------------+
|total_records|distinct_quarters|minimum_ren

location_code,location,quarter_code,quarter,quarter_start_date,average_monthly_rent_eur
120500,Dublin,20074,2007Q4,2007-10-01,1312.69
120500,Dublin,20081,2008Q1,2008-01-01,1330.34
120500,Dublin,20082,2008Q2,2008-04-01,1324.99
120500,Dublin,20083,2008Q3,2008-07-01,1286.49
120500,Dublin,20084,2008Q4,2008-10-01,1255.69
120500,Dublin,20091,2009Q1,2009-01-01,1226.66
120500,Dublin,20092,2009Q2,2009-04-01,1186.17
120500,Dublin,20093,2009Q3,2009-07-01,1144.57
120500,Dublin,20094,2009Q4,2009-10-01,1096.30
120500,Dublin,20101,2010Q1,2010-01-01,1066.61



RENTAL VALUES BY PROPERTY TYPE
------------------------------------------------------------


location_code,location,quarter,quarter_start_date,property_type_code,property_type,average_monthly_rent_eur
120500,Dublin,2025Q4,2025-10-01,01,Detached house,3057.95
120500,Dublin,2025Q4,2025-10-01,02,Semi detached house,2598.88
120500,Dublin,2025Q4,2025-10-01,03,Terrace house,2432.52
120500,Dublin,2025Q4,2025-10-01,04,Apartment,2154.18
120500,Dublin,2025Q4,2025-10-01,05,Other flats,1444.97



RENTAL VALUES BY BEDROOM CATEGORY
------------------------------------------------------------


location_code,location,quarter,quarter_start_date,bedroom_code,bedroom_category,average_monthly_rent_eur
120500,Dublin,2025Q4,2025-10-01,08,Four plus bed,3319.36
120500,Dublin,2025Q4,2025-10-01,03,Three bed,2585.87
120500,Dublin,2025Q4,2025-10-01,02,Two bed,2253.28
120500,Dublin,2025Q4,2025-10-01,07,1 to 3 bed,2100.45
120500,Dublin,2025Q4,2025-10-01,06,1 to 2 bed,2014.24
120500,Dublin,2025Q4,2025-10-01,01,One bed,1740.68


In [0]:
trends = spark.table(
    "workspace.dublin_rental_market.gold_dublin_rent_trends"
)

ordered_trends = trends.orderBy("quarter_start_date")

initial_rent = (
    ordered_trends
    .first()["average_monthly_rent_eur"]
)

latest_rent = (
    ordered_trends
    .orderBy(F.col("quarter_start_date").desc())
    .first()["average_monthly_rent_eur"]
)

growth = ((latest_rent / initial_rent) - 1) * 100

print(f"Initial rent: €{initial_rent:.2f}")
print(f"Latest rent: €{latest_rent:.2f}")
print(f"Total nominal growth: {growth:.2f}%")

Initial rent: €1312.69
Latest rent: €2165.02
Total nominal growth: 64.93%


##  Data Quality - Automated Validation

This section implements automated data quality checks across the Bronze, Silver, and Gold layers. The tests validate record counts, dimensional uniqueness, rental values, reporting periods, and analytical table consistency.

In [0]:
from pyspark.sql import functions as F

# Load Delta tables
bronze = spark.table(
    "workspace.dublin_rental_market.bronze_rental_observations"
)

silver = spark.table(
    "workspace.dublin_rental_market.silver_rental_observations"
)

gold = spark.table(
    "workspace.dublin_rental_market.gold_dublin_rent_trends"
)

# Define data quality tests
quality_checks = [
    (
        "Bronze record count",
        bronze.count() == 1_367_436
    ),
    (
        "Silver record count",
        silver.count() == 1_367_436
    ),
    (
        "No duplicate dimensional combinations",
        silver.groupBy(
            "statistic_code",
            "quarter_code",
            "bedroom_code",
            "property_type_code",
            "location_code"
        ).count().filter(F.col("count") > 1).count() == 0
    ),
    (
        "No non-positive rental values",
        silver.filter(
            F.col("average_monthly_rent_eur").isNotNull()
            & (F.col("average_monthly_rent_eur") <= 0)
        ).count() == 0
    ),
    (
        "No invalid reporting periods",
        silver.filter(
            F.col("year").isNull()
            | F.col("quarter_number").isNull()
            | F.col("quarter_start_date").isNull()
        ).count() == 0
    ),
    (
        "Gold contains only valid rental values",
        gold.filter(
            F.col("average_monthly_rent_eur").isNull()
            | (F.col("average_monthly_rent_eur") <= 0)
        ).count() == 0
    ),
    (
        "Dublin historical series contains 73 quarters",
        gold.select("quarter").distinct().count() == 73
    )
]

# Display test results
for test_name, passed in quality_checks:
    status = "PASS" if passed else "FAIL"
    print(f"{status}: {test_name}")

# Final summary
passed_tests = sum(result for _, result in quality_checks)
total_tests = len(quality_checks)

print("\nDATA QUALITY SUMMARY")
print(f"Passed: {passed_tests}/{total_tests}")
print(f"Failed: {total_tests - passed_tests}/{total_tests}")

PASS: Bronze record count
PASS: Silver record count
PASS: No duplicate dimensional combinations
PASS: No non-positive rental values
PASS: No invalid reporting periods
PASS: Gold contains only valid rental values
PASS: Dublin historical series contains 73 quarters

DATA QUALITY SUMMARY
Passed: 7/7
Failed: 0/7


##  Data Quality - Dynamic Validation

This section improves the validation framework by replacing hardcoded source counts with dynamically calculated expectations. It also verifies missing-value consistency and reconciliation between Silver and Gold analytical data.


In [0]:
from pyspark.sql import functions as F

# Load data layers
bronze = spark.table(
    "workspace.dublin_rental_market.bronze_rental_observations"
)

silver = spark.table(
    "workspace.dublin_rental_market.silver_rental_observations"
)

gold = spark.table(
    "workspace.dublin_rental_market.gold_dublin_rent_trends"
)

# Calculate dynamic expectations
expected_records = len(observations)

bronze_count = bronze.count()
silver_count = silver.count()

expected_dublin_quarters = (
    silver
    .filter(
        (F.col("location_code") == "120500") &
        (F.col("bedroom_code") == "-") &
        (F.col("property_type_code") == "-") &
        (F.col("is_valid_rent") == True)
    )
    .select("quarter")
    .distinct()
    .count()
)

# Check dimensional uniqueness
duplicate_count = (
    silver
    .groupBy(
        "statistic_code",
        "quarter_code",
        "bedroom_code",
        "property_type_code",
        "location_code"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Define automated quality checks
quality_checks = [
    ("Bronze matches source record count",
     bronze_count == expected_records),

    ("Silver matches source record count",
     silver_count == expected_records),

    ("No duplicate dimensional combinations",
     duplicate_count == 0),

    ("No non-positive rental values",
     silver.filter(
         F.col("average_monthly_rent_eur").isNotNull()
         & (F.col("average_monthly_rent_eur") <= 0)
     ).count() == 0),

    ("Missing value flags are consistent",
     silver.filter(
         (F.col("is_missing_rent") != F.col("average_monthly_rent_eur").isNull())
     ).count() == 0),

    ("Valid value flags are consistent",
     silver.filter(
         (F.col("is_valid_rent") != (
             F.col("average_monthly_rent_eur").isNotNull()
             & (F.col("average_monthly_rent_eur") > 0)
         ))
     ).count() == 0),

    ("No invalid reporting periods",
     silver.filter(
         F.col("year").isNull()
         | F.col("quarter_number").isNull()
         | F.col("quarter_start_date").isNull()
         | (~F.col("quarter_number").between(1, 4))
     ).count() == 0),

    ("Gold contains only valid rental values",
     gold.filter(
         F.col("average_monthly_rent_eur").isNull()
         | (F.col("average_monthly_rent_eur") <= 0)
     ).count() == 0),

    ("Gold reconciles with Silver",
     gold.select("quarter").distinct().count()
     == expected_dublin_quarters)
]

# Display results
for test_name, passed in quality_checks:
    status = "PASS" if passed else "FAIL"
    print(f"{status}: {test_name}")

# Final summary
passed_tests = sum(result for _, result in quality_checks)
total_tests = len(quality_checks)

print("\nDATA QUALITY SUMMARY")
print(f"Passed: {passed_tests}/{total_tests}")
print(f"Failed: {total_tests - passed_tests}/{total_tests}")

PASS: Bronze matches source record count
PASS: Silver matches source record count
PASS: No duplicate dimensional combinations
PASS: No non-positive rental values
PASS: Missing value flags are consistent
PASS: Valid value flags are consistent
PASS: No invalid reporting periods
PASS: Gold contains only valid rental values
PASS: Gold reconciles with Silver

DATA QUALITY SUMMARY
Passed: 9/9
Failed: 0/9
